# Notebook 3 — Model Training

This notebook trains two deep-learning architectures on the FER2013 dataset
and saves the resulting models and training histories to disk.

- **Model A — Custom CNN**: a four-block convolutional network trained from
  scratch on 48 × 48 grayscale images. Serves as the interpretable baseline.
- **Model B — MobileNetV2 fine-tuned**: a lightweight pretrained backbone
  adapted to FER2013 in two stages. Expected to generalise better due to
  ImageNet pretraining.

All evaluation (accuracy curves, confusion matrices, per-class metrics) is
deferred to Notebook 4. Here we only run a brief smoke test after each model
is saved to confirm the output pipeline works end-to-end.

## Section 1 — Setup

We fix all random seeds before any other operation to ensure reproducibility
across runs. Seeds are set for Python's `random` module, NumPy, and TensorFlow
independently because each maintains its own PRNG state. We also verify GPU
availability here; training without a GPU is possible but wall-clock time
increases roughly 10×.

In [ ]:
import json
import random
import sys
from pathlib import Path

import numpy as np
import tensorflow as tf

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config
from src.models.cnn_custom import build_cnn_custom
from src.models.mobilenet_finetune import build_mobilenet_head, unfreeze_top_layers
from src.models.trainer import get_default_callbacks, save_history, train_model

SEED = config.RANDOM_SEED
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print(f"GPU available: {gpus}")
else:
    print("WARNING: No GPU detected. Training will be significantly slower.")